---
title: "Convolutional Neural Networks (Part 2)"
short_title: Part 2
subject: DEEP LEARNING
---

## Introduction

In this notebook, we use **transfer learning** and **fine-tune** ResNet18 pretrained on ImageNet for a binary image classification task on the [Histopathologic Cancer Detection](https://www.kaggle.com/competitions/histopathologic-cancer-detection/data) dataset. The objective is to identify metastatic tissue from histopathologic scans of lymph node sections ({numref}`03-histopathologic-cancer-detection`). To train the large network with relatively few training examples, we perform **data augmentation** replacing an input $\textbf{\textsf{x}}$ with $T_\theta(\textbf{\textsf{x}})$ where $T_\theta$ is a label-preserving stochastic transformation.

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = "svg"
from okt.nn.utils import get_device, set_seed

from tqdm import tqdm
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import torch
import torch.nn as nn
import torch.nn.functional as F

import random
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

ROOT_DIR = Path("../../").resolve()
LOCAL_DIR = ROOT_DIR / ".local/"
DATASET_DIR = LOCAL_DIR / "data/"
ARTIFACTS_DIR = LOCAL_DIR / "artifacts/"
warnings.simplefilter(action="ignore")
matplotlib.rcParams["image.interpolation"] = "nearest"

RANDOM_SEED = 0
DEVICE = get_device()
print(f"Using device: {DEVICE}")
set_seed(RANDOM_SEED)

Using device: mps
seed: 0  deterministic: False


**NOTE:** The folder structure of the [downloaded dataset](https://www.kaggle.com/docs/api#interacting-with-datasets) should be:
```
./data/histopathologic-cancer-detection
├── test
├── train
└── train_labels.csv
```

In [2]:
import cv2

IMG_DATASET_DIR = DATASET_DIR / "histopathologic-cancer-detection"
data = pd.read_csv(IMG_DATASET_DIR / "train_labels.csv")

fig, ax = plt.subplots(3, 5, figsize=(6, 4.5))
for k in range(15):
    i, j = divmod(k, 5)
    fname = str(IMG_DATASET_DIR / "train" / f"{data.id[k]}.tif")
    ax[i, j].imshow(cv2.imread(fname))
    ax[i, j].set_title(data.label[k], size=10)
    ax[i, j].axis("off")
fig.tight_layout()

plt.savefig("./plots/03-histopathologic-cancer-detection.svg", bbox_inches="tight")
plt.close("all")

:::{figure} ./plots/03-histopathologic-cancer-detection.svg
---
name: 03-histopathologic-cancer-detection
width: 100%
align: center
---
**Sample images from the dataset.** 1 = the center 32×32 region of a patch contains at least one pixel of tumor tissue. Hence, tumor tissue in the outer region of the patch does not influence the label. This outer region is provided to enable fully-convolutional models that do not use zero-padding, to ensure consistent behavior when applied to a whole-slide image. 0 = otherwise.
:::


## Data augmentation

Data augmentation incorporates **transformed** or **perturbed** versions of the original images into the dataset. More precisely, each data point
$(\textbf{\textsf{x}}, y)$ in a mini-batch is replaced by $(T_{\theta}(\textbf{\textsf{x}}), y)$ during training
where $T_{\theta}$ is a stochastic label-preserving transformation. At inference, an input $\textbf{\textsf{x}}$ is replaced by $\mathbb{E}[T_{\theta}(\textbf{\textsf{x}})].$ 

These properties are satisfied by the ff:

In [3]:
from torchvision import transforms

transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),      # rot. angle ~ U[-20, +20] deg.
    transforms.CenterCrop([49, 49]),    # static. 49 >= 32
])

transform_infer = transforms.Compose([
    transforms.ToTensor(),
    transforms.CenterCrop([49, 49]),
])

Transforms are applied at each mini-batch sample using CPU (e.g. set `num_workers > 0` in the data loader):

In [4]:
from torch.utils.data import DataLoader, Dataset, Subset

class HistopathologicDataset(Dataset):
    def __init__(self, data, train=True, transform=None):
        split = "train" if train else "test"
        self.fnames = [str(IMG_DATASET_DIR / split / f"{fn}.tif") for fn in data.id]
        self.labels = data.label.tolist()
        self.transform = transform
    
    def __len__(self):
        return len(self.fnames)
    
    def __getitem__(self, index):
        img = cv2.imread(self.fnames[index])
        if self.transform:
            img = self.transform(img)
        
        return img, self.labels[index]


data = data.sample(frac=0.3)
split = int(0.80 * len(data))
ds_train = HistopathologicDataset(data[:split], train=True, transform=transform_train)
ds_valid = HistopathologicDataset(data[split:], train=True, transform=transform_infer)

Some imbalance (nothing too severe):

In [5]:
# % positive class
data[:split].label.mean(), data[split:].label.mean()

(np.float64(0.4060334052948529), np.float64(0.40842296621723984))

Simulating images across epochs:

In [6]:
simul_train = DataLoader(Subset(ds_train, torch.arange(3)), batch_size=3, shuffle=True)
simul_valid = DataLoader(Subset(ds_valid, torch.arange(1)), batch_size=1, shuffle=False)

In [7]:
fig, ax = plt.subplots(3, 4)
for e in range(3):
    img_train, tgt_train = next(iter(simul_train))
    for i in range(3):
        if i == 0:
            ax[e, i].set_ylabel(f"Epoch: {e}")
        
        img, tgt = img_train[i], tgt_train[i]
        ax[e, i].imshow(img.permute(1, 2, 0).detach())
        ax[e, i].set_xlabel(tgt.item())
        ax[e, i].set_xticks([])
        ax[e, i].set_yticks([])
        ax[0, i].set_title(f"instance: {i}")

    img_valid, tgt_valid = next(iter(simul_valid))
    ax[e, 3].set_xlabel(tgt_valid[0].item())
    ax[e, 3].imshow(img_valid[0].permute(1, 2, 0).detach())
    ax[e, 3].set_xticks([])
    ax[e, 3].set_yticks([])

ax[0, 3].set_title("valid")
fig.tight_layout()

plt.savefig("./plots/03-histopathologic-cancer-detection-dataloader.svg", bbox_inches="tight")
plt.close("all")

:::{figure} ./plots/03-histopathologic-cancer-detection-dataloader.svg
---
name: 03-histopathologic-cancer-detection-dataloader
width: 100%
align: center
---
**Simulating a mini-batch (B = 3).**
Training instances are stochastically transformed at each epoch. Meanwhile, test instances have fixed transformations. Note that labels are not affected by the transforms.** 
:::


## Transfer learning

Transfer learning is a technique[^hf] used to leverage large models trained on a generic related task (i.e. the **pretrained model**). In this notebook, we use ResNet {cite:p}`resnet` which is trained to classify  [ImageNet](https://image-net.org/) consisting of 1M+ images in 1000 categories. To adapt the pretrained model to our task, we retain only the feature extractors and train a new **classification head** ({numref}`03-transfer-learning`).

To avoid nullifying the pretrained weights with large random gradients, we first have to train the classification head to convergence, while keeping the weights of the pretrained model fixed. Then, we proceed with **fine-tuning** where we train the entire model with a very low learning rate, again so that the pretrained weights are gradually changed.

[^hf]: More than a training technique, transfer learning enables a paradigm of computation reuse (compute is an expensive resource) and fosters collaboration across the ML community. See [HuggingFace models](https://huggingface.co/models) platform where fine-tuning and base models are tracked.

:::{figure} ./img/03-transfer-learning.png
---
name: 03-transfer-learning
width: 100%
align: center
---
**Transfer learning.** Feature extractors learned from a foundational task (e.g. classification and localization on [ImageNet](https://www.image-net.org/challenges/LSVRC/index.php)) is adapted to the new task. This may require removing part of the original network and replacing it with one that is more appropriate to the current task. **Source:** {cite:p}`keras2{Fig 8.12}`.
:::

PyTorch conveniently provides a collection of [vision models](https://pytorch.org/vision/main/models.html) with pretrained weights:

In [8]:
import torchinfo
from torchvision import models

resnet = models.resnet18(pretrained=True)

BATCH_SIZE = 16
torchinfo.summary(resnet, input_size=(BATCH_SIZE, 3, 49, 49))

Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [16, 1000]                --
├─Conv2d: 1-1                            [16, 64, 25, 25]          9,408
├─BatchNorm2d: 1-2                       [16, 64, 25, 25]          128
├─ReLU: 1-3                              [16, 64, 25, 25]          --
├─MaxPool2d: 1-4                         [16, 64, 13, 13]          --
├─Sequential: 1-5                        [16, 64, 13, 13]          --
│    └─BasicBlock: 2-1                   [16, 64, 13, 13]          --
│    │    └─Conv2d: 3-1                  [16, 64, 13, 13]          36,864
│    │    └─BatchNorm2d: 3-2             [16, 64, 13, 13]          128
│    │    └─ReLU: 3-3                    [16, 64, 13, 13]          --
│    │    └─Conv2d: 3-4                  [16, 64, 13, 13]          36,864
│    │    └─BatchNorm2d: 3-5             [16, 64, 13, 13]          128
│    │    └─ReLU: 3-6                    [16, 64, 13, 13]          --
│

As described above, features from the pretrained model are passsed to a new classifier[^1]:

[^1]: Batch normalization {cite:p}`batchnorm` helps with activation and gradient stability. Dropout regularizes both the incoming pretrained features as well as the hidden layer.

In [9]:
in_features = resnet.fc.in_features
num_hidden = 256

head = nn.Sequential(
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.BatchNorm1d(in_features),
    nn.Dropout(0.5),
    nn.Linear(in_features, num_hidden),
    nn.ReLU(),
    nn.BatchNorm1d(num_hidden),
    nn.Dropout(0.5),
    nn.Linear(num_hidden, 2),
)

model = nn.Sequential(
    nn.Sequential(*list(resnet.children())[:-2]),
    head
)

### Frozen features

Freezing the feature extraction layers:

In [10]:
for param in model[0].parameters(): # model[0] = pretrained
    param.requires_grad = False

Setting up the dataloaders[^2]:

[^2]: Recall from the previous notebook that the Dataset object applies the transform function at each indexing call (e.g. when sampling a mini-batch). Applying `Subset` gets a subset of the data based on the provided index set. Here, a contiguous index set is fine since the data has been pre-shuffled. Hence, we have the same set of images, but are augmented at each training step.

In [11]:
train_loader = DataLoader(Subset(ds_train, torch.arange(32000)), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(Subset(ds_valid, torch.arange(8000)),  batch_size=BATCH_SIZE, shuffle=False)

Training the model using AdamW {cite:p}`adamw` with learning rate `0.001`:

In [12]:
from okt.nn.cnn import Trainer

Trainer??

In [13]:
from torch.optim.lr_scheduler import OneCycleLR

epochs = 2
optim = torch.optim.AdamW(model.parameters(), lr=0.001)
scheduler = OneCycleLR(optim, max_lr=0.01, steps_per_epoch=len(train_loader), epochs=epochs)
trainer = Trainer(model, optim, loss_fn=F.cross_entropy, scheduler=scheduler, device=DEVICE)
trainer.run(epochs=epochs, train_loader=train_loader, valid_loader=valid_loader)

100%|██████████| 2/2 [02:49<00:00, 84.99s/it]



[Epoch: 1/2]    loss: 0.5979  acc: 0.6806    val_loss: 0.5232  val_acc: 0.7455
[Epoch: 2/2]    loss: 0.5800  acc: 0.6906    val_loss: 0.5014  val_acc: 0.7698


In [14]:
trainer.plot_training_history(annotate=True, markersize=18)
plt.savefig("./plots/03-histopathologic-cancer-detection-training-history.svg", bbox_inches="tight")
plt.close("all")

:::{figure} ./plots/03-histopathologic-cancer-detection-training-history.svg
---
name: 03-histopathologic-cancer-detection-training-history
width: 100%
align: center
---
**Classifier head trained with frozen base.** The pretrained base acts as feature extractor while the classifier weights are updated away from its random initialization. Here training loss and accuracy are averaged over the last few steps. The validation loss and accuracy are computed at the end of each epoch over the entire validation dataset&mdash; simulating inference performance if we load the trained model at that checkpoint. Train metrics are accumulated from each mini-batch to save compute. Wait... is that the fabled double descent curve? ( ˶°ㅁ°)
:::

### Fine-tuning

Unfreezing the pretrained model layers. Note that we set small learning rates:

In [15]:
for param in model[0].parameters():
    param.requires_grad = True

# 100x smaller lr (both optim and scheduler)
epochs = 5
optim = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
scheduler = OneCycleLR(optim, max_lr=0.0001, steps_per_epoch=len(train_loader), epochs=epochs)
trainer_ft = Trainer(model, optim, loss_fn=F.cross_entropy, scheduler=scheduler, device=DEVICE)
trainer_ft.run(epochs=epochs, train_loader=train_loader, valid_loader=valid_loader)

100%|██████████| 5/5 [17:02<00:00, 204.42s/it]



[Epoch: 1/5]    loss: 0.4304  acc: 0.8113    val_loss: 0.3820  val_acc: 0.8359
[Epoch: 2/5]    loss: 0.4036  acc: 0.8206    val_loss: 0.3608  val_acc: 0.8458
[Epoch: 3/5]    loss: 0.3562  acc: 0.8481    val_loss: 0.3208  val_acc: 0.8679
[Epoch: 4/5]    loss: 0.3316  acc: 0.8575    val_loss: 0.3021  val_acc: 0.8820
[Epoch: 5/5]    loss: 0.3151  acc: 0.8794    val_loss: 0.2949  val_acc: 0.8852


In [16]:
m = len(trainer.train_log["loss"])
trainer.train_log["loss"] += trainer_ft.train_log["loss"]
trainer.train_log["accs"] += trainer_ft.train_log["accs"]
trainer.train_log["loss_avg"] += trainer_ft.train_log["loss_avg"]
trainer.train_log["accs_avg"] += trainer_ft.train_log["accs_avg"]
trainer.valid_log["loss"] += trainer_ft.valid_log["loss"]
trainer.valid_log["accs"] += trainer_ft.valid_log["accs"]

fig, ax = trainer.plot_training_history(annotate=True, markersize=15)
ax[0].axvline(m, color="C3", linestyle="dashed", label="[fine-tuning]", alpha=0.8)
ax[1].axvline(m, color="C3", linestyle="dashed", label="[fine-tuning]", alpha=0.8)
ax[1].legend()

plt.savefig("./plots/03-histopathologic-cancer-detection-fine-tuning.svg", bbox_inches="tight")
plt.close("all");

:::{figure} ./plots/03-histopathologic-cancer-detection-fine-tuning.svg
---
name: 03-histopathologic-cancer-detection-fine-tuning
width: 100%
align: center
---
**Fine-tuning the model.** Recall that the weights of the classification head are trained on the outputs of the frozen pretrained model. After the classification head forms proper weights, the pretrained weights are unfreezed, and trained with small LR. Performance improves rapidly once the base weights are unfreezed, then we get steady improvement.
:::

**Remarks.** Model overfits immediately and validation curves diverge when data augmentation is turned off (i.e. memorization). Dense layers also do not train well without BN. Finally, the dataset is slightly imbalanced, so accuracy may not be the best metric to use (though not very important for our purposes).

### Model inference

Converting the data loaders for batch inference:

In [17]:
from okt.nn.utils import eval_context

eval_context??

In [18]:
@torch.inference_mode()
def batch_predict(trainer: Trainer, input_loader: DataLoader):
    with eval_context(trainer.model):
        preds = [trainer(x) for x in input_loader]
        preds = torch.cat(preds, dim=0)
    return preds


class InputDataLoader(DataLoader):
    """Extract input from training data loaders."""
    
    def __init__(self, data_loader: DataLoader):
        self.data_loader = data_loader

    def __iter__(self):
        for inp, tgt in self.data_loader:
            yield inp

pred = batch_predict(trainer, InputDataLoader(valid_loader))
print(pred.shape)
print(pred)

torch.Size([8000, 2])
tensor([[-1.0409,  1.0232],
        [ 1.1303, -1.2083],
        [-1.1200,  1.1271],
        ...,
        [ 1.4929, -1.5238],
        [-0.2759,  0.1703],
        [ 0.2845, -0.3899]], device='mps:0')


This should be equal to the final validation accuracy:

In [19]:
y = torch.cat([tgt for _, tgt in valid_loader], dim=0)
print((pred.argmax(dim=1) == y.to(DEVICE)).float().mean().item())
print(trainer_ft.evaluate(valid_loader)["accs"])
print(trainer_ft.valid_log["accs"][-1]) # or look at final valid log

0.8852500319480896
0.88525
0.88525


**NOTE:** The input from our dataloaders come transformed. For processing raw images, we have to transform the inputs in eval mode:

In [20]:
filepath = IMG_DATASET_DIR / "test/0a0a1f3867f41e02353afcaf503f63be1bdd35ec.tif"
data = cv2.imread(filepath)
x = transform_infer(data).unsqueeze(0)

print(trainer.predict(x))

tensor([[ 1.9674, -1.9618]], device='mps:0')


In [21]:
PATH = ARTIFACTS_DIR / "cancer_detection_model.pkl"
torch.save(trainer.model.state_dict(), PATH)

---